In [1]:
import os
import sys


PROJECT_ROOT = '/content/drive/MyDrive/Plagiarism-Detection-System'

os.chdir(PROJECT_ROOT)
print(f"Current Working Directory: {os.getcwd()}")
sys.path.append(PROJECT_ROOT)

Current Working Directory: /content/drive/MyDrive/Plagiarism-Detection-System


In [2]:
!pip install openai-whisper torchaudio omegaconf einops nnAudio
!pip install umap-learn plotly pandas pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 48.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.7 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=ff6d2ccb84ecf53372a4c722daf3d34bd321ef04e8aaf5d9f26bb4ed97bfa23f
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [2]:
import os

%cd /content/drive/MyDrive/Plagiarism-Detection-System/

!mkdir -p models/clews
!mkdir -p models/wealy

if not os.path.exists("models/wealy/checkpoint.pt"):
    print("Downloading WEALY (from Hugging Face)...")
    !wget -O models/wealy/checkpoint.pt https://huggingface.co/audio-based-lyrics-matching/wealy-whisper-dvi/resolve/main/checkpoint_best.ckpt
else:
    print("WEALY model already exists. Skipping download.")

if not os.path.exists("models/clews/checkpoint.pt"):
    print("\nDownloading CLEWS (from Zenodo)...")
    !wget -O clews_checkpoints.zip "https://zenodo.org/records/15045900/files/clews.zip?download=1"
    !unzip clews_checkpoints.zip -d downloaded_checkpoints

    !find downloaded_checkpoints -name "*best.ckpt" | grep "dvi" | head -n 1 | xargs -I {} cp {} models/clews/checkpoint.pt

    !if [ ! -f models/clews/checkpoint.pt ]; then find downloaded_checkpoints -name "*.ckpt" | head -n 1 | xargs -I {} cp {} models/clews/checkpoint.pt; fi

    !rm clews_checkpoints.zip
    !rm -rf downloaded_checkpoints
else:
    print("\nCLEWS model already exists. Skipping download.")

print("\nSaved Models Successfully!")

/content
WEALY model already exists. Skipping download.

CLEWS model already exists. Skipping download.

Saved Models Successfully!


In [ ]:
print("Starting extraction for CLEWS...")
!PYTHONPATH=. python src/inference/extract_clews.py
print("Success!")

Starting extraction for CLEWS...
Loading existing parquet file: data/clews_embeddings.parquet
Found 7200 already processed files.
Remaining files to process: 7428
Extracting CLEWS embeddings:   3% 199/7428 [03:19<1:59:57,  1.00it/s]
[Checkpoint] Saved 7400 embeddings so far...
Extracting CLEWS embeddings:   5% 399/7428 [06:30<1:46:42,  1.10it/s]
[Checkpoint] Saved 7600 embeddings so far...
Extracting CLEWS embeddings:   8% 599/7428 [09:41<2:16:36,  1.20s/it]
[Checkpoint] Saved 7800 embeddings so far...
Extracting CLEWS embeddings:  11% 799/7428 [12:51<1:49:51,  1.01it/s]
[Checkpoint] Saved 8000 embeddings so far...
Extracting CLEWS embeddings:  13% 999/7428 [15:54<1:44:05,  1.03it/s]
[Checkpoint] Saved 8200 embeddings so far...
Extracting CLEWS embeddings:  16% 1199/7428 [19:02<1:31:33,  1.13it/s]
[Checkpoint] Saved 8400 embeddings so far...
Extracting CLEWS embeddings:  19% 1399/7428 [22:11<1:38:22,  1.02it/s]
[Checkpoint] Saved 8600 embeddings so far...
Extracting CLEWS embeddings:  

In [ ]:
print("Starting extraction for WEALY...")
!PYTHONPATH=. python src/inference/extract_wealy.py
print("Success!")

Starting extraction for WEALY...
Loading Whisper model on cuda...
Initializing WEALY model...
/content/drive/MyDrive/Plagiarism-Detection-System/src/utils/wealy_lib.py:75: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=conf.num_transformer_blocks)
Remaining files to process: 14628
Extracting WEALY embeddings:  11% 1620/14628 [20:08<2:41:41,  1.34it/s]

In [ ]:
print("Starting Metric Learning...")
!PYTHONPATH=. python src/evaluation/metrics.py --model all
print("Success!")

Starting Metric Learning...
Calculating distances for CLEWS...
Traceback (most recent call last):
  File "/content/drive/MyDrive/Plagiarism-Detection-System/src/evaluation/metrics.py", line 218, in <module>
    compute_distances(CLEWS_PARQUET, SMP_CSV, CLEWS_RESULTS)
  File "/content/drive/MyDrive/Plagiarism-Detection-System/src/evaluation/metrics.py", line 156, in compute_distances
    emb_mod_np = np.array(row['embedding_mod'].tolist(), dtype=np.float32)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'float' object has no attribute 'tolist'
Success!


In [ ]:
print("Executing Late Fusion...")
!PYTHONPATH=. python src/evaluation/late_fusion_metrics.py
print("Success!")